In [ ]:
import os
from tensorflow import keras
import tensorflow as tf
from keras import layers, models
import numpy as np
import kagglehub
import pandas as pd

In [ ]:
# Download latest version
path = kagglehub.dataset_download("shayanfazeli/heartbeat")

In [ ]:
train_csv_path = os.path.join(path, "mitbih_train.csv")
test_csv_path = os.path.join(path, "mitbih_test.csv")

In [ ]:
df_train = pd.read_csv(train_csv_path, header=None).astype("float32")
df_test = pd.read_csv(test_csv_path, header=None).astype("float32")

# Il dataset ha le righe in ordine raggruppate per classe
# Quando vado a separare train e val con validation=0.2...
# mette in validation solo classi di un tipo (se va bene)
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# MAPPING DELLE CLASSI
classes = ['N', 'S', 'V', 'F', 'Q']

In [ ]:
# Imposto le features e le etichette/targets
X_train = df_train.iloc[:, :-1].values # features tutte le colonne tranne l'ultima
y_train = df_train.iloc[:, -1].values # target solo l'ultima colonna

In [ ]:
# converto in 3D per LSTM
X_train_3D = np.expand_dims(X_train, axis=2) # --> (87554, 187, 1)

In [ ]:
# MODELLO
model = keras.Sequential([
    layers.Input(shape=(187, 1)), # 187 features e 1 sensore (battito)

    # LSTM Layer
    layers.LSTM(64, return_sequences=False), # False perchè dopo non c'è una LSTM
    layers.Dropout(0.2),

    # Layer di uscita per la regressione (1 valore: la temperatura del giorno dopo)
    layers.Dense(5, activation="softmax")
])

In [ ]:
# COMPILAZIONE
# model is a classifier with 5 classes. it needs of optimizer, loss function and
# metric accuracy
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# TRAINING

hystory = model.fit(X_train_3D,
                    y_train,
                    epochs=5,
                    batch_size=32,
                    validation_split=0.2)

In [ ]:
# PREPARAZIONE TEST

# prende solo le features perchè anche nel test file di questo dataset
# c'è la colonna labels
X_test = df_test.iloc[:, :-1].values

# converto anche il test in 3D
X_test_3D = np.expand_dims(X_test, axis=2)

In [ ]:
# CLASSIFICAZIONE
predictions = model.predict(X_test_3D)

In [ ]:
# ESTRAZIONE DATI DALLA PREDIZIONE

# calcola la probabilità e prende il massimo per riga (axis 1 cioè orizzontale)
predicted_classes = np.argmax(predictions, axis=1)

# converte le etichette numeriche in lettere comprensibili
predicted_labels = np.array(classes)[predicted_classes]

print("Classi previste per i primi 10 battiti:", predicted_labels[:10])

In [87]:
# VERIFICO CHE LE PREDIZIONI SIANO DIVERSE TRA LORO
print(pd.Series(predicted_labels).value_counts())

N    19196
Q     1570
V     1086
S       40
Name: count, dtype: int64
